# Traceprop-LLM — head-to-head overhead vs. LogIX (Choe et al.)

**Question (MLSys reviewer, item 12):** getting per-sample gradients from forward/backward hooks is the same trick LogIX (github.com/logix-project/logix) and Opacus use. How does inline logging via `LoRAGradientLogger` compare to running LogIX's own hooks inline, on the same model?

**CPU proxy result already run (tiny synthetic classifier, same tracked scope, 100 repeats each):**

| | overhead |
|---|---|
| LogIX | 12.9% ± 19.3% |
| LoRAGradientLogger | 25.9% ± 14.7% |

LogIX was *faster* on this small CPU model — held at both n=20 and n=100 repeats, so it's not pure noise. But this is the opposite regime from the paper's actual claim (GPT-2/Pythia on an L4 GPU, ~1% overhead); LogIX's memory-mapped I/O and compression machinery is built for exactly that larger regime and may have more fixed small-model overhead that disappears at scale. This notebook runs the real comparison: same model, same tracked scope, same batch/seq as `exp25`.

**Note:** `logix-ai` requires Python <3.11. Check `!python --version` in the cell below — if Colab's default is 3.11+, you'll need `!pip install logix-ai` to fail loudly (expected) and then set up a 3.10 kernel, or use `!apt install python3.10` + a venv. Colab's default runtime is usually 3.10 or 3.11 depending on the image; check first.

In [ ]:
!python --version
!pip -q install transformers peft datasets accelerate logix-ai
!pip -q uninstall -y torchao 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
import logix; print('logix ok:', logix.__file__)

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!pip -q install -e .

## Traceprop's own overhead (exp25) -- run this first for the baseline

This is the number already in the paper (Table 1). Re-run it here so both numbers come from the same session/GPU/instant, not two different Colab runs with different thermal/contention state.

In [ ]:
%cd /content/Traceprop/experiments
!python exp25_llm_inline_overhead.py --backend hf --model gpt2 --device cuda \
    --steps 200 --repeats 20 --track 1 --proj_dim 512

## LogIX's overhead, same model/scope (exp31)

In [ ]:
%cd /content/Traceprop/experiments
!python exp31_logix_comparison.py --backend hf --model gpt2 --device cuda \
    --steps 200 --repeats 20 --track 1

## Result

Compare `overhead_pct_median` from both runs. If Traceprop is still cheaper at this scale (opposite of the CPU proxy), that's the real item-12 answer: write it up as "comparable mechanism, lower overhead at the scale that matters" and name the specific implementation choices (on-device projection kept off the critical path via `buffer=True`/`drain()`, sparse-JL vs LogIX's PCA/covariance-based compression, no disk I/O in the hot path) as the reason. If LogIX is still cheaper here too, report that honestly and reframe the contribution around what's novel structurally (source-file lineage integration, not raw speed).

In [ ]:
import json, glob
print('--- Traceprop (exp25) ---')
for p in glob.glob('/content/Traceprop/experiments/results/exp25_hf_*.json'):
    d = json.load(open(p))
    print(d.get('model'), 'throughput_overhead_pct', d.get('throughput_overhead_pct'), '+/-', d.get('throughput_overhead_std'))
print('--- LogIX (exp31) ---')
for p in glob.glob('/content/Traceprop/experiments/results/exp31_logix_hf_*.json'):
    d = json.load(open(p))
    print(d.get('model'), 'overhead_pct_median', d.get('overhead_pct_median'), '+/-', d.get('overhead_pct_std'))